In [9]:
# Limit BLAS/OpenMP threads to avoid libomp/libiomp clashes and reduce oversubscription
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'


In [10]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, module="threadpoolctl")

In [11]:
# Disable user site packages for this kernel session to avoid ABI mismatches
import os, sys, site
os.environ['PYTHONNOUSERSITE'] = '1'
try:
    usp = site.getusersitepackages()
except Exception:
    usp = None
if usp and isinstance(usp, str):
    sys.path = [p for p in sys.path if not p.startswith(usp)]
print('User site disabled. Removed from sys.path if present.')


User site disabled. Removed from sys.path if present.


# Med3D ➜ LoCalPFN (Fast & Progress)

This variant is optimized for quicker iteration and includes progress/timing. It uses smaller K, a smaller TabPFN ensemble, and reduced PCA dimensionality.

In [12]:
# Imports and project path setup
from pathlib import Path
import sys, json, time

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'med3pipe').exists():
    if (PROJECT_ROOT.parent / 'med3pipe').exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

from med3pipe import (
    # Steps 1–3
    prepare_for_sam3d, split_validation, Sam3DPaths, find_default_sam3d_root,
    # Steps 4–6
    build_sam3d_model, default_feature_dirs, extract_embeddings_train_val,
    load_roi_features, load_labels_from_sheet, build_y,
    # Step 7
    standardize_pca,
)
from med3pipe.tabular.localpfn import (
    build_knn_index, retrieve_neighbors, _logit, _apply_adapter, _train_adapter, _auto_k
)
from med3pipe.tabular.tabpfn import ensure_tabpfn_on_sys_path
ensure_tabpfn_on_sys_path(None)
from tabpfn.classifier import TabPFNClassifier  # type: ignore

device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
device_str


Project root: c:\Users\cahel\Desktop\Med3Tab-PFN


'cpu'

## Parameters (fast defaults)

In [13]:
# Dataset
from pathlib import Path
DATASET_ROOT = Path('gist')
CATEGORY = 'gist'
CT_NAME = 'ct_GIST'
CASE_GLOB = None
MAX_CASES = None

SPLIT_RATIO = 0.8
SEED = 2025

# SAM-Med3D model
MODEL_TYPE = 'vit_b_ori'
CHECKPOINT = None
IMG_SIZE = 128

# Labels
DATASET_NAME = 'GIST'
SUBJECT_COL = 'Subject'
LABEL_COL = 'Diagnosis_binary'
CASE_SUFFIX = '_CT'

# LoCalPFN
K = 16                       # small for speed; raise to 32/64 after sanity check
FIT_ADAPTER = False           # start without adapter to reduce runtime
ADAPTER_EPOCHS = 5
ADAPTER_LR = 5e-2
ADAPTER_WEIGHT_DECAY = 0.0
ADAPTER_NUM_QUERIES = 64      # if enabling adapter later
RETRIEVAL_METRIC = 'euclidean'

# Reduce TabPFN compute for faster fits
TABPFN_KW = {} # 4 or 8; raise later for best accuracy

# Dimensionality reduction
N_COMPONENTS_MAX = 128        # reduce PCA dimension for faster fits

# Validate on a subset first to confirm flow
FAST_VALIDATE = True
VAL_LIMIT = 10                 # first N val samples; set FAST_VALIDATE=False to run all

(K, FIT_ADAPTER, TABPFN_KW, N_COMPONENTS_MAX, FAST_VALIDATE, VAL_LIMIT)


(16, False, {}, 128, True, 10)

## Steps 1–2: Prepare dataset for SAM-Med3D

In [14]:
SAM3D_ROOT = find_default_sam3d_root()
if SAM3D_ROOT is None:
    raise RuntimeError('Could not auto-detect SAM-Med3D root. Please set SAM3D_ROOT manually.')

prepared, paths = prepare_for_sam3d(
    dataset_root=(PROJECT_ROOT / DATASET_ROOT),
    sam3d_root=SAM3D_ROOT,
    category=CATEGORY,
    ct_name=CT_NAME,
    case_glob=CASE_GLOB,
    max_cases=MAX_CASES,
)
prepared, paths


Prepared 25 cases ...
Prepared 50 cases ...
Prepared 75 cases ...
Prepared 100 cases ...
Prepared 125 cases ...
Prepared 150 cases ...
Prepared 175 cases ...
Prepared 200 cases ...
Prepared 225 cases ...
Done. Prepared 246 cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST


(246,
 Sam3DPaths(sam3d_root=WindowsPath('C:/Users/cahel/Desktop/Med3Tab-PFN/SAM-Med3D-main/SAM-Med3D-main'), category='gist', ct_name='ct_GIST'))

## Step 3: Create validation split

In [15]:
ntr, nval = split_validation(paths, split_ratio=SPLIT_RATIO, seed=SEED, copy=True)
print('Split done | Train:', ntr, '| Val:', nval)
paths.train_root, paths.val_root


Validation set copied to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST | Train: 196 | Val: 50
Split done | Train: 196 | Val: 50


(WindowsPath('C:/Users/cahel/Desktop/Med3Tab-PFN/SAM-Med3D-main/SAM-Med3D-main/data/train/gist/ct_GIST'),
 WindowsPath('C:/Users/cahel/Desktop/Med3Tab-PFN/SAM-Med3D-main/SAM-Med3D-main/data/validation/gist/ct_GIST'))

## Step 4: Build SAM-Med3D encoder and extract embeddings

In [16]:
model = build_sam3d_model(
    sam3d_root=SAM3D_ROOT,
    model_type=MODEL_TYPE,
    checkpoint=CHECKPOINT,
    device=torch.device(device_str),
    eval_mode=True,
)
feat_dirs = default_feature_dirs(SAM3D_ROOT, category=CATEGORY, ct_name=CT_NAME)
extract_embeddings_train_val(paths, model, sam3d_root=SAM3D_ROOT, img_size=IMG_SIZE, feature_dirs=feat_dirs, device=torch.device(device_str))
feat_dirs


To extract: 246 from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST\imagesTr
Done extracting to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\features\gist\ct_GIST_train
To extract: 78 from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST\imagesVal
Done extracting to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\features\gist\ct_GIST


FeatureDirs(train_dir=WindowsPath('C:/Users/cahel/Desktop/Med3Tab-PFN/SAM-Med3D-main/SAM-Med3D-main/features/gist/ct_GIST_train'), val_dir=WindowsPath('C:/Users/cahel/Desktop/Med3Tab-PFN/SAM-Med3D-main/SAM-Med3D-main/features/gist/ct_GIST'))

## Step 5: ROI-pooled features per case

In [17]:
X_train, ids_train = load_roi_features(feat_dirs.train_dir, paths.labels_tr)
X_val, ids_val = load_roi_features(feat_dirs.val_dir, paths.labels_val)
X_train.shape, X_val.shape, len(ids_train), len(ids_val)


((246, 384), (78, 384), 246, 78)

## Step 6: Labels and alignment to case IDs

In [18]:
sheet_csv = (PROJECT_ROOT / 'gist' / 'sheet.csv')
if not sheet_csv.exists():
    candidates = [
        PROJECT_ROOT / 'gist' / 'sheet.csv',
        Path.cwd() / 'gist' / 'sheet.csv',
        SAM3D_ROOT.parent.parent / 'gist' / 'sheet.csv',
    ]
    sheet_csv = next((c for c in candidates if c.exists()), None)
if sheet_csv is None or not Path(sheet_csv).exists():
    raise FileNotFoundError('sheet.csv not found; please set the path above explicitly.')

df, lab_map = load_labels_from_sheet(
    sheet_csv=Path(sheet_csv),
    dataset_name=DATASET_NAME,
    subject_col=SUBJECT_COL,
    label_col=LABEL_COL,
    case_suffix=CASE_SUFFIX,
)
y_train, _ = build_y(ids_train, lab_map)
y_val, _ = build_y(ids_val, lab_map)
y_train.shape, y_val.shape


((246,), (78,))

## Step 7: Standardize + PCA (fit on TRAIN, apply to VAL)

In [19]:
X_train_p, X_val_p, scaler, pca = standardize_pca(
    X_train=X_train,
    X_val=X_val,
    n_components_max=N_COMPONENTS_MAX,
    random_state=42,
)
X_train_p.shape, X_val_p.shape


((246, 128), (78, 128))

## Step 8: Retrieval, optional adapter training, and local-context inference (with progress)

In [20]:
# Choose k (use implemented _auto_k if K is None)
k_eff = int(K) if K is not None else int(_auto_k(X_train_p.shape[0]))
print('Effective k:', k_eff)

# Build KNN index on TRAIN (PCA space)
t0 = time.time(); knn = build_knn_index(X_train_p, metric=RETRIEVAL_METRIC); print('KNN fit:', f'{time.time()-t0:.3f}s')

# Optional adapter training (kept here, defaults off)
adapter = None
if FIT_ADAPTER:
    print('[LoCalPFN-fast] Training adapter on local neighborhoods...')
    n_q = min(int(ADAPTER_NUM_QUERIES), X_train_p.shape[0])
    rng = np.random.default_rng(42)
    q_idx = rng.choice(X_train_p.shape[0], size=n_q, replace=False)
    logits_list, y_list = [], []
    clf = TabPFNClassifier(device=device_str, **TABPFN_KW)
    t_adapt = time.time()
    for qi in q_idx:
        idxs, _ = retrieve_neighbors(knn, X_train_p[qi:qi+1], k=k_eff)
        neigh = idxs[0]
        neigh = neigh[neigh != qi]
        if neigh.size == 0:
            continue
        X_ctx, y_ctx = X_train_p[neigh], y_train[neigh]
        clf.fit(X_ctx, y_ctx)
        proba = clf.predict_proba(X_train_p[qi:qi+1])
        if proba.shape[1] != 2:
            continue
        logit = _logit(proba[:, 1:2])
        logits_list.append(logit[0, 0])
        y_list.append(int(y_train[qi]))
    if len(logits_list) > 10:
        logits_arr = np.array(logits_list, dtype=np.float32)
        y_arr = np.array(y_list, dtype=np.float32)
        adapter = _train_adapter(
            logits=logits_arr, y=y_arr, epochs=ADAPTER_EPOCHS, lr=ADAPTER_LR, weight_decay=ADAPTER_WEIGHT_DECAY, verbose=True
        )
        print('Adapter trained in', f'{time.time()-t_adapt:.1f}s')
    else:
        print('[LoCalPFN-fast] Skipping adapter training (insufficient samples).')

# Inference with progress
print('[LoCalPFN-fast] Running local-context inference on validation set...')
t_all = time.time()
idxs_val, _ = retrieve_neighbors(knn, X_val_p, k=k_eff)
if FAST_VALIDATE:
    idxs_val = idxs_val[:VAL_LIMIT]
clf = TabPFNClassifier(device=device_str, **TABPFN_KW)
y_pred = np.zeros((idxs_val.shape[0],), dtype=np.int64)
proba_buf = []
for i, neigh in enumerate(idxs_val):
    t_i = time.time()
    X_ctx, y_ctx = X_train_p[neigh], y_train[neigh]
    clf.fit(X_ctx, y_ctx)
    try:
        p = clf.predict_proba(X_val_p[i:i+1])  # (1, C)
        if adapter is not None and p.shape[1] == 2:
            z = _logit(p[:, 1:2])
            z_adj = _apply_adapter(z, adapter)
            p1 = 1.0 / (1.0 + np.exp(-z_adj))
            p = np.concatenate([1 - p1, p1], axis=1)
        proba_buf.append(p[0])
    except Exception:
        pass
    y_pred[i] = int(clf.predict(X_val_p[i:i+1])[0])
    if (i + 1) % 5 == 0 or (i + 1) == len(idxs_val):
        print('[{}/{}] iter {:.2f}s | total {:.1f}s'.format(i+1, len(idxs_val), time.time()-t_i, time.time()-t_all), flush=True)

print('VAL local inference took:', f'{time.time()-t_all:.1f}s')

proba_val = None
if len(proba_buf) == len(y_pred):
    proba_val = np.stack(proba_buf, axis=0)

acc = float(accuracy_score(y_val[:len(y_pred)], y_pred))
macro_f1 = float(f1_score(y_val[:len(y_pred)], y_pred, average='macro'))
roc_auc = None
if proba_val is not None and proba_val.shape[1] == 2:
    roc_auc = float(roc_auc_score(y_val[:len(y_pred)], proba_val[:, 1]))
print('Accuracy:', acc)
print('Macro F1:', macro_f1)
print('ROC AUC:', roc_auc)

print(classification_report(y_val[:len(y_pred)], y_pred, target_names=['0','1']))
confusion_matrix(y_val[:len(y_pred)], y_pred)


Effective k: 16
KNN fit: 0.003s
[LoCalPFN-fast] Running local-context inference on validation set...
[5/10] iter 14.11s | total 61.7s
[10/10] iter 12.61s | total 125.1s
VAL local inference took: 125.1s
Accuracy: 1.0
Macro F1: 1.0
ROC AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         8

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10



array([[2, 0],
       [0, 8]])

# One-shot Med3-LocalPFNmethod 

In [ ]:
# Minimal one-shot LoCalPFN (no cleanup)
from pathlib import Path
import time

from med3pipe import find_default_sam3d_root, local_end_to_end, LocalPFNConfig

SAM3D_ROOT = find_default_sam3d_root()

# Fast LoCalPFN config
cfg = LocalPFNConfig(
    k=K,
    device=device_str,          # 'cuda' if available; else 'cpu'
    fit_adapter=FIT_ADAPTER,
    adapter_epochs=ADAPTER_EPOCHS,
    clf_kwargs={},              # keep empty for compatibility with your TabPFN
)

# Unique out_dir so artifacts never overwrite previous runs
out_dir = Path("tabpfn_runs") / f"local_{CATEGORY}_{CT_NAME}_{int(time.time())}"

# One-shot run (prepare -> split -> embeddings -> ROI -> labels -> PCA -> LoCalPFN)
# Omit sheet_csv to auto-resolve: <dataset_root>/sheet.csv or common fallbacks
res_e2e = local_end_to_end(
    dataset_root=(PROJECT_ROOT / DATASET_ROOT),
    category=CATEGORY,
    ct_name=CT_NAME,
    case_glob=CASE_GLOB,
    max_cases=MAX_CASES,            # you can set this small (e.g., 60) for faster iterations
    split_ratio=SPLIT_RATIO,        # increase (e.g., 0.9–0.95) to reduce VAL size
    seed=SEED,
    sam3d_root=SAM3D_ROOT,
    model_type=MODEL_TYPE,
    checkpoint=CHECKPOINT,
    img_size=IMG_SIZE,
    device=device_str,
    sheet_csv=None,                 # let pipeline auto-find sheet.csv
    dataset_name=DATASET_NAME,
    subject_col=SUBJECT_COL,
    label_col=LABEL_COL,
    case_suffix=CASE_SUFFIX,
    n_components_max=N_COMPONENTS_MAX,
    random_state=42,
    local_out_dir=out_dir,
    local_cfg=cfg,
)

# Path to saved metrics
res_e2e.localpfn["metrics_path"]

Clean slate ready.
Prepared 25 cases ...
Prepared 50 cases ...
Prepared 75 cases ...
Prepared 100 cases ...
Prepared 125 cases ...
Prepared 150 cases ...
Prepared 175 cases ...
Prepared 200 cases ...
Prepared 225 cases ...
Done. Prepared 246 cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST
Validation set copied to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST | Train: 196 | Val: 50
To extract: 246 from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST\imagesTr
Extracted 25 embeddings ...
Extracted 50 embeddings ...
Extracted 75 embeddings ...
Extracted 100 embeddings ...
Extracted 125 embeddings ...
Extracted 150 embeddings ...
Extracted 175 embeddings ...
Extracted 200 embeddings ...
Extracted 225 embeddings ...
Done extracting to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\features\gist\ct_GIST_train
To extract: 50 from C:\User

WindowsPath('tabpfn_runs/local_gist_ct_GIST_1758197930/localpfn_metrics.json')